# 多模态共享LoRA梯度冲突分析（大文件优化版）

这个版本针对 `gradient_logs.jsonl` **很大** 的场景做了优化：

- 不再一次性 `read_json` 全量加载
- 每个实验模块按需加载所需列（`usecols`）
- 支持按 step 范围过滤与采样率（`step_mod`）
- 公共清洗逻辑在加载器内完成，模块之间互不依赖全局大 DataFrame

> 建议先设置 `STEP_MIN / STEP_MAX / STEP_MOD`，快速迭代图表；最终出图再放开限制。


In [ ]:
import ast
import json
from pathlib import Path
from typing import Callable, Iterable, Optional

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")
pd.set_option('display.max_columns', 200)


In [ ]:
# ===== 配置区（大文件优化） =====
LOG_PATH = Path('gradient_logs.jsonl')
OUTPUT_DIR = Path('analysis_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 快速调试建议：先限制 step 范围和抽样，再逐步放开
STEP_MIN = None      # 如 0
STEP_MAX = None      # 如 2000
STEP_MOD = None      # 如 5 -> 每5步取1步

assert LOG_PATH.exists(), f'文件不存在: {LOG_PATH.resolve()}'
print('Using log file:', LOG_PATH.resolve())


In [ ]:
def _parse_modality_ids(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            y = json.loads(x)
            return y if isinstance(y, list) else []
        except Exception:
            try:
                y = ast.literal_eval(x)
                return y if isinstance(y, list) else []
            except Exception:
                return []
    return []


def _step_pass(step_val, step_min=None, step_max=None, step_mod=None):
    try:
        s = int(step_val)
    except Exception:
        return False
    if step_min is not None and s < step_min:
        return False
    if step_max is not None and s > step_max:
        return False
    if step_mod is not None and step_mod > 1 and (s % step_mod != 0):
        return False
    return True


def load_jsonl_subset(
    path: Path,
    usecols: Optional[Iterable[str]] = None,
    partition_in: Optional[set] = None,
    step_min=None,
    step_max=None,
    step_mod=None,
    row_predicate: Optional[Callable[[dict], bool]] = None,
):
    rows = []
    usecols = set(usecols) if usecols is not None else None

    with path.open('r', encoding='utf-8') as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            # step过滤（先过滤，减少后续开销）
            step_val = obj.get('step', None)
            if step_val is not None and not _step_pass(step_val, step_min, step_max, step_mod):
                continue

            if partition_in is not None:
                p = obj.get('grad_partition', 'all')
                if p not in partition_in:
                    continue

            if row_predicate is not None and not row_predicate(obj):
                continue

            if usecols is None:
                row = obj
            else:
                row = {k: obj.get(k, None) for k in usecols}

            rows.append(row)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    # 统一清洗
    if 'modality_ids' in df.columns:
        df['modality_ids'] = df['modality_ids'].apply(_parse_modality_ids)

    for c in ['step','layer_depth','grad_norm','grad_mean','grad_std','grad_abs_mean','numel','supervised_token_count']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    if 'grad_was_none' in df.columns:
        df['grad_was_none'] = df['grad_was_none'].fillna(False).astype(bool)

    if 'grad_partition' in df.columns:
        df['grad_partition'] = df['grad_partition'].fillna('all')

    return df


## 一、数据质量与覆盖度检查（轻量）

仅加载最少字段：`step, grad_partition, layer_depth, param_name, grad_was_none`。

In [ ]:
df_meta = load_jsonl_subset(
    LOG_PATH,
    usecols=['step','grad_partition','layer_depth','param_name','grad_was_none'],
    step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
)

summary = {
    'rows_loaded': len(df_meta),
    'step_min': int(df_meta['step'].min()) if len(df_meta)>0 and 'step' in df_meta else None,
    'step_max': int(df_meta['step'].max()) if len(df_meta)>0 and 'step' in df_meta else None,
    'n_layers': int(df_meta['layer_depth'].nunique()) if 'layer_depth' in df_meta else 0,
    'n_params': int(df_meta['param_name'].nunique()) if 'param_name' in df_meta else 0,
    'partitions': sorted(df_meta['grad_partition'].dropna().unique().tolist()) if 'grad_partition' in df_meta else [],
    'none_ratio': float(df_meta['grad_was_none'].mean()) if 'grad_was_none' in df_meta and len(df_meta)>0 else None,
}
summary


In [ ]:
if len(df_meta) > 0:
    plt.figure(figsize=(8,4))
    order = ['all', 'text_only', 'image_only']
    ax = sns.countplot(data=df_meta, x='grad_partition', order=[p for p in order if p in df_meta['grad_partition'].unique()])
    ax.set_title('Partition Record Count')
    ax.set_xlabel('grad_partition')
    ax.set_ylabel('count')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_partition_count.png', dpi=180)
    plt.show()
else:
    print('No records loaded under current step filter.')


## 二、层级-分区梯度强度热力图（按需加载）

仅加载：`layer_depth, grad_partition, grad_norm`。

In [ ]:
df_lp = load_jsonl_subset(
    LOG_PATH,
    usecols=['layer_depth','grad_partition','grad_norm','step'],
    step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
)

if len(df_lp) > 0:
    layer_partition = df_lp.groupby(['layer_depth','grad_partition'], as_index=False)['grad_norm'].mean()
    pivot_lp = layer_partition.pivot(index='layer_depth', columns='grad_partition', values='grad_norm').sort_index()

    plt.figure(figsize=(10,8))
    sns.heatmap(pivot_lp, cmap='magma', cbar_kws={'label':'mean grad_norm'})
    plt.title('Layer vs Partition Mean Grad Norm')
    plt.xlabel('grad_partition')
    plt.ylabel('layer_depth')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_layer_partition_gradnorm_heatmap.png', dpi=200)
    plt.show()
else:
    print('No data for layer-partition heatmap.')


## 三、step趋势图（按需加载）

仅加载：`step, grad_partition, grad_norm`。

In [ ]:
df_trend = load_jsonl_subset(
    LOG_PATH,
    usecols=['step','grad_partition','grad_norm'],
    step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
)

if len(df_trend) > 0:
    trend = df_trend.groupby(['step','grad_partition'], as_index=False)['grad_norm'].mean()
    plt.figure(figsize=(12,5))
    sns.lineplot(data=trend, x='step', y='grad_norm', hue='grad_partition', linewidth=2)
    plt.title('Mean Grad Norm over Steps by Partition')
    plt.xlabel('step')
    plt.ylabel('mean grad_norm')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_step_partition_gradnorm_trend.png', dpi=180)
    plt.show()
else:
    print('No data for trend plot.')


## 四、主导指数分析（text vs image）

仅加载子分区需要的字段：`step, param_name, layer_depth, adapter_type, grad_partition, grad_norm`。

In [ ]:
df_dom = load_jsonl_subset(
    LOG_PATH,
    usecols=['step','param_name','layer_depth','adapter_type','grad_partition','grad_norm'],
    partition_in={'text_only','image_only'},
    step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
)

if len(df_dom) > 0:
    key_cols = ['step','param_name','layer_depth','adapter_type']
    wide = df_dom.pivot_table(index=key_cols, columns='grad_partition', values='grad_norm', aggfunc='mean').reset_index()
    wide = wide.dropna(subset=['text_only','image_only'])

    eps = 1e-12
    wide['dominance_index'] = (wide['image_only'] - wide['text_only']) / (wide['image_only'] + wide['text_only'] + eps)

    plt.figure(figsize=(11,5))
    sns.histplot(wide['dominance_index'], bins=60, kde=True)
    plt.axvline(0, color='red', linestyle='--', linewidth=1.5)
    plt.title('Dominance Index Distribution (image vs text)')
    plt.xlabel('dominance_index')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_dominance_hist.png', dpi=180)
    plt.show()

    dom_layer = wide.groupby('layer_depth', as_index=False)['dominance_index'].mean().sort_values('layer_depth')
    plt.figure(figsize=(11,5))
    sns.lineplot(data=dom_layer, x='layer_depth', y='dominance_index', marker='o')
    plt.axhline(0, color='red', linestyle='--', linewidth=1.2)
    plt.title('Layer-wise Mean Dominance Index')
    plt.xlabel('layer_depth')
    plt.ylabel('mean dominance_index')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_layer_dominance.png', dpi=180)
    plt.show()

    display(wide[['dominance_index']].describe())
else:
    print('No data for dominance analysis.')


## 五、`grad_was_none` 机制诊断（按需加载）

仅加载：`layer_depth, grad_partition, grad_was_none`。

In [ ]:
df_none = load_jsonl_subset(
    LOG_PATH,
    usecols=['layer_depth','grad_partition','grad_was_none','step'],
    step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
)

if len(df_none) > 0:
    none_stat = df_none.groupby(['layer_depth','grad_partition'], as_index=False)['grad_was_none'].mean()
    pivot_none = none_stat.pivot(index='layer_depth', columns='grad_partition', values='grad_was_none').sort_index()

    plt.figure(figsize=(10,8))
    vmax = float(np.nanmax(pivot_none.values)) if np.isfinite(np.nanmax(pivot_none.values)) else 1.0
    sns.heatmap(pivot_none, cmap='Blues', vmin=0, vmax=max(0.01, vmax), cbar_kws={'label':'none ratio'})
    plt.title('Layer vs Partition None-Gradient Ratio')
    plt.xlabel('grad_partition')
    plt.ylabel('layer_depth')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_layer_partition_none_ratio.png', dpi=200)
    plt.show()
else:
    print('No data for none-gradient diagnosis.')


## 六、All 与子分区一致性检查（按需加载）

只读取 `all/text_only/image_only` 所需字段，并在 merge 前先做 groupby 聚合减少数据量。

In [ ]:
df_cons = load_jsonl_subset(
    LOG_PATH,
    usecols=['step','param_name','layer_depth','grad_partition','grad_norm'],
    partition_in={'all','text_only','image_only'},
    step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
)

if len(df_cons) > 0:
    # 先聚合减少 join 规模
    agg = df_cons.groupby(['step','param_name','layer_depth','grad_partition'], as_index=False)['grad_norm'].mean()

    all_df = agg[agg['grad_partition']=='all'][['step','param_name','layer_depth','grad_norm']].rename(columns={'grad_norm':'all_norm'})
    txt_df = agg[agg['grad_partition']=='text_only'][['step','param_name','layer_depth','grad_norm']].rename(columns={'grad_norm':'text_norm'})
    img_df = agg[agg['grad_partition']=='image_only'][['step','param_name','layer_depth','grad_norm']].rename(columns={'grad_norm':'image_norm'})

    chk = all_df.merge(txt_df, on=['step','param_name','layer_depth'], how='inner')                .merge(img_df, on=['step','param_name','layer_depth'], how='inner')

    if len(chk) > 0:
        chk['sub_sum'] = chk['text_norm'] + chk['image_norm']
        chk['all_minus_subsum'] = chk['all_norm'] - chk['sub_sum']

        plt.figure(figsize=(11,5))
        sns.histplot(chk['all_minus_subsum'], bins=60, kde=True)
        plt.axvline(0, color='red', linestyle='--')
        plt.title('all_norm - (text_norm + image_norm)')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'fig_all_vs_subsum_diff.png', dpi=180)
        plt.show()

        display(chk[['all_norm','text_norm','image_norm','sub_sum','all_minus_subsum']].describe())
    else:
        print('No overlap records among all/text_only/image_only after aggregation.')
else:
    print('No data for consistency check.')


## 七、论文表格导出（按需加载）

仅加载生成统计表所需字段。

In [ ]:
df_tbl = load_jsonl_subset(
    LOG_PATH,
    usecols=['layer_depth','grad_partition','grad_norm','grad_was_none','supervised_token_count','step'],
    step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
)

if len(df_tbl) > 0:
    table_layer_partition = (
        df_tbl.groupby(['layer_depth','grad_partition'])
              .agg(mean_grad_norm=('grad_norm','mean'),
                   std_grad_norm=('grad_norm','std'),
                   mean_none_ratio=('grad_was_none','mean'),
                   mean_token_count=('supervised_token_count','mean'))
              .reset_index()
    )
    table_path = OUTPUT_DIR / 'table_layer_partition_stats.csv'
    table_layer_partition.to_csv(table_path, index=False, encoding='utf-8-sig')
    print('Saved:', table_path.resolve())
    display(table_layer_partition.head(10))
else:
    print('No data for table export.')


## 八、如果要做更深层机制分析，还需补充什么？

基于你现在日志（统计量）还建议补充：

1. **梯度向量存储**（分层抽样）  
   - 新增 `grad_vector_path`（或每N步存一次向量）才能精确算 `cos(text,image)` 与SVD容量。  
2. **激活统计**  
   - 记录同层 hidden norm / variance，可把“梯度冲突”与“表示异质性”直接关联。  
3. **任务难度与batch构成控制变量**  
   - 记录 batch 内 text/image/video 比例、token数、loss，做偏相关分析。  
4. **时间窗口稳健性**  
   - 对 step 做 rolling 分析，避免瞬时噪声误导结论。


## 九、可选扩展：当日志包含 `grad_vector_path`

只有在你后续保存了梯度向量（`.npy`）后才可运行此模块。

In [ ]:
df_vec_meta = load_jsonl_subset(
    LOG_PATH,
    usecols=['step','param_name','layer_depth','grad_partition','grad_vector_path'],
    partition_in={'text_only','image_only'},
    step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
)

if 'grad_vector_path' in df_vec_meta.columns and df_vec_meta['grad_vector_path'].notna().any():
    import numpy.linalg as LA

    sample = df_vec_meta.dropna(subset=['grad_vector_path']).head(500).copy()
    key = ['step','param_name','layer_depth']

    txt = sample[sample['grad_partition']=='text_only'][key+['grad_vector_path']].rename(columns={'grad_vector_path':'text_path'})
    img = sample[sample['grad_partition']=='image_only'][key+['grad_vector_path']].rename(columns={'grad_vector_path':'img_path'})
    pair = txt.merge(img, on=key, how='inner')

    cos_vals = []
    for _, r in pair.iterrows():
        vt = np.load(Path(r['text_path'])).reshape(-1)
        vi = np.load(Path(r['img_path'])).reshape(-1)
        cos_vals.append(float(np.dot(vt, vi) / (LA.norm(vt)*LA.norm(vi) + 1e-12)))

    pair['cosine'] = cos_vals
    display(pair['cosine'].describe())

    plt.figure(figsize=(8,4))
    sns.histplot(pair['cosine'], bins=40, kde=True)
    plt.title('Cosine(text_grad, image_grad)')
    plt.tight_layout()
    plt.show()
else:
    print('当前日志未包含可用 grad_vector_path，跳过余弦/SVD精确分析。')
